In [1]:
# Auto-generated by cram CLI — do not hand-edit.
# Rebuild with: python tools/cram.py build

import sys
from pathlib import Path

# Portably discover repo root containing 'engine' and add to sys.path
_cur = Path.cwd().resolve()
while _cur != _cur.parent:
    if (_cur / 'engine' / '__init__.py').is_file():
        if str(_cur) not in sys.path:
            sys.path.insert(0, str(_cur))
        break
    _cur = _cur.parent

# In WebAssembly / Pyodide / JupyterLite environments, pre-load dependencies
if 'pyodide' in sys.modules or hasattr(sys, '_emscripten_info') or sys.platform == 'emscripten':
    try:
        import piplite
        await piplite.install(['pyyaml', 'matplotlib', 'ipywidgets', 'numpy'])
    except Exception:
        pass

# Explicit imports for Pyodide AST scanner and interactive widget ecosystem
try:
    import yaml
except ImportError:
    pass
import matplotlib.pyplot as plt
import ipywidgets as widgets

from engine import Worksheet, apply_default_style

apply_default_style()

ws = Worksheet.load('lesson_07_analytical_force_derivatives.yaml')
ws.start()


$\def\AA{\text{Å}}$

# MSE 485 · Lesson 07: Analytical Force Derivatives — From Bond Stretching to Dihedral Torsions

_Session 13c1794e · Estimated time: ~110 minutes · 38 problems_

**Recording your answers.** Submit each problem as you go. Your answers are saved to the workspace as you submit, *and* the running record at the bottom of the notebook keeps your progress. When you finish, run the last two cells — **`show_record()`** and **`finalize()`** — to record your participation. **If you reopened this workspace** (or the kernel restarted), use **Kernel → Restart Kernel and Run All Cells** before finalizing, so every answer is reloaded.

### Learning Objectives
- Prove why Molecular Dynamics requires exact analytical force derivatives F_i = -nabla_{r_i} E to conserve total energy and linear/angular momentum.
- Derive 2-body radial forces for bond-stretching and non-bonded pair potentials using the multivariable chain rule and unit vector gradients.
- Derive 3-body bond-bending (angle) forces via the arccos chain rule and vector quotient rule, and prove why terminal forces are strictly perpendicular to the bond vectors.
- Analyze 4-body dihedral (torsion) potentials in the OPLS Fourier form and apply the singularity-free Blondel-Karplus plane-normal derivative formulas.
- Verify analytical force implementations using central finite-difference gradient testing and identify the optimal perturbation step delta balancing truncation and roundoff error.


In [2]:
ws.render_section('sec_intro_why_analytical_forces')

## 1. Why Molecular Dynamics Demands Exact Analytical Forces

In Molecular Dynamics (MD) and structural relaxation, an interatomic potential (force field) defines a scalar potential energy surface $E_{\text{total}}(\{\mathbf{r}_1, \mathbf{r}_2, \dots, \mathbf{r}_N\})$ as a function of the $3N$ atomic coordinates. In all-atom molecular force fields such as **OPLS-AA** (*Optimized Potentials for Liquid Simulations — All Atom*), the total potential energy is decomposed into covalent bonded and non-bonded terms:
$$E_{\text{total}} = \sum_{\text{bonds}} K_r (r - r_{\text{eq}})^2 + \sum_{\text{angles}} K_\theta (\theta - \theta_{\text{eq}})^2 + \sum_{\text{torsions}} \sum_{n=1}^{3} \frac{V_n}{2}\bigl(1 + \cos(n\phi + f_n)\bigr) + \sum_{a < b} \sum_{i \in a} \sum_{j \in b} \left[ \frac{q_i q_j e^2}{r_{ij}} + 4\epsilon_{ij}\left(\left(\frac{\sigma_{ij}}{r_{ij}}\right)^{12} - \left(\frac{\sigma_{ij}}{r_{ij}}\right)^6\right) \right]$$

To integrate Newton's equations of motion $m_i \ddot{\mathbf{r}}_i = \mathbf{F}_i$, we need the **exact negative gradient** of the total energy with respect to each atom's 3D position vector $\mathbf{r}_i = (x_i, y_i, z_i)$:
$$\mathbf{F}_i = -\nabla_{\mathbf{r}_i} E_{\text{total}} = -\left( \frac{\partial E_{\text{total}}}{\partial x_i}\hat{\mathbf{x}} + \frac{\partial E_{\text{total}}}{\partial y_i}\hat{\mathbf{y}} + \frac{\partial E_{\text{total}}}{\partial z_i}\hat{\mathbf{z}} \right)$$

### Conservation Laws and Computational Cost
Why not simply approximate $\mathbf{F}_i$ numerically by nudging each coordinate by $\pm\delta$?
1. **Total Energy Conservation in NVE MD:** Taking the time derivative of total mechanical energy $H = K + E_{\text{total}} = \sum_{i=1}^N \frac{1}{2}m_i \dot{\mathbf{r}}_i^2 + E_{\text{total}}(\{\mathbf{r}_i\})$ along a trajectory yields:
   $$\frac{dH}{dt} = \sum_{i=1}^N m_i \dot{\mathbf{r}}_i \cdot \ddot{\mathbf{r}}_i + \sum_{i=1}^N \nabla_{\mathbf{r}_i} E_{\text{total}} \cdot \dot{\mathbf{r}}_i = \sum_{i=1}^N \bigl(\mathbf{F}_i + \nabla_{\mathbf{r}_i} E_{\text{total}}\bigr) \cdot \dot{\mathbf{r}}_i$$
   If $\mathbf{F}_i$ differs from $-\nabla_{\mathbf{r}_i} E_{\text{total}}$ even slightly (e.g., due to missing chain-rule terms or finite-difference noise), $\mathbf{F}_i$ is no longer a conservative vector field, causing systematic **total energy drift** over thousands of MD steps.
2. **Computational Scaling:** Evaluating $E_{\text{total}}$ by central finite differences requires $2 \times 3N = 6N$ separate energy evaluations per timestep (or $3N+1$ for forward differences). By contrast, applying the **analytical multivariable chain rule** computes all $3N$ force components simultaneously in a single pass with $\mathcal{O}(1)$ overhead relative to a single energy evaluation!
3. **Translational and Rotational Invariance:** Because internal coordinates (distances $r_{ij}$, bond angles $\theta_{ijk}$, and dihedral angles $\phi_{ijkl}$) depend only on **relative position vectors** $\mathbf{r}_{ij} = \mathbf{r}_i - \mathbf{r}_j$, shifting the entire system by a uniform vector $\mathbf{c}$ leaves $E_{\text{total}}$ unchanged. Differentiating $E_{\text{total}}(\{\mathbf{r}_i + \mathbf{c}\}) = E_{\text{total}}(\{\mathbf{r}_i\})$ with respect to $\mathbf{c}$ at $\mathbf{c}=\mathbf{0}$ proves that the net internal force on any isolated molecule or sub-cluster must vanish identically:
   $$\sum_i \mathbf{F}_i = -\sum_i \nabla_{\mathbf{r}_i} E_{\text{total}} = \mathbf{0}, \qquad \sum_i \mathbf{r}_i \times \mathbf{F}_i = \mathbf{0}$$


In [3]:
ws.render_section('q01_energy_conservation_condition')

**Q1 · easy** · 💡

Along a classical Molecular Dynamics trajectory satisfying $m_i \ddot{\mathbf{r}}_i = \mathbf{F}_i$, the instantaneous rate of change of total mechanical energy $H = \sum_i \frac{1}{2}m_i \dot{\mathbf{r}}_i^2 + E_{\text{total}}(\{\mathbf{r}_i\})$ is given by $\frac{dH}{dt} = \sum_i (\mathbf{F}_i + \nabla_{\mathbf{r}_i} E_{\text{total}})\cdot\dot{\mathbf{r}}_i$. What happens in a microcanonical (NVE) simulation if a programmer omits one chain-rule term when coding $\mathbf{F}_i$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check whether the quantity inside the parentheses vanishes for arbitrary atomic velocities when F_i != -nabla_{r_i} E_total.

</div>
</details>

- [ ] **(A)** The force field becomes non-conservative ($\mathbf{F}_i \ne -\nabla_{\mathbf{r}_i} E_{\text{total}}$), causing $dH/dt \ne 0$ and producing systematic drift in the total energy over time.
- [ ] **(B)** The velocity Verlet integrator automatically projects out the non-conservative component of $\mathbf{F}_i$, so total energy is still exactly conserved.
- [ ] **(C)** Only the kinetic temperature fluctuates, while the total energy H remains constant to machine precision.
- [ ] **(D)** The simulation remains conservative as long as the timestep $\Delta t$ is reduced by a factor of 2.

In [4]:
ws.render_section('q02_opls_force_field_terms')

**Q2 · easy** · 💡

In the OPLS-AA all-atom potential energy expression, which of the following statements correctly match each potential term to the geometric internal coordinate on which it directly depends? *(Select all that apply.)*


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Inspect the four summations in the OPLS-AA energy formula: bond-stretching, bond-bending, torsion, and non-bonded interactions.

</div>
</details>

- [ ] **(A)** Both covalent bond-stretching $E_{\text{bond}}$ and non-bonded (Coulomb + Lennard-Jones) interactions depend strictly on 2-body interatomic distances $r_{ij} = |\mathbf{r}_i - \mathbf{r}_j|$.
- [ ] **(B)** Bond-bending $E_{\text{angle}} = \sum K_\theta (\theta - \theta_{\text{eq}})^2$ is a 3-body interaction depending on the angle $\theta_{123}$ formed by two bond vectors sharing a central vertex atom.
- [ ] **(C)** Torsion (dihedral) energy $E(\phi)$ is a 4-body interaction depending on the angle $\phi$ between the (1-2-3) and (2-3-4) atomic planes along a 4-atom chain.
- [ ] **(D)** Lennard-Jones van der Waals interactions require 4-body cross-product normals to compute atomic forces.

In [5]:
ws.render_section('q03_translational_invariance_sum')

**Q3 · medium** · 💡

Because any internal potential energy term $E(\{\mathbf{r}_i\})$ (whether a 2-body bond, 3-body angle, or 4-body dihedral) is invariant under a rigid translation $\mathbf{r}_i \to \mathbf{r}_i + \mathbf{c}$ of all participating atoms, what algebraic identity must the analytical gradients satisfy for an $m$-atom interaction ($m \in \{2, 3, 4\}$)?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Differentiate E(r_1 + c, ..., r_m + c) = E(r_1, ..., r_m) with respect to the translation vector c using the multivariable chain rule.

</div>
</details>

- [ ] **(A)** $\sum_{i=1}^m \nabla_{\mathbf{r}_i} E = 0$, so the internal forces satisfy $\sum_{i=1}^m \mathbf{F}_i = 0$ (allowing one atom's force to be obtained for free from the negative sum of the others).
- [ ] **(B)** $\sum_{i=1}^m |\nabla_{\mathbf{r}_i} E|^2 = 0$, implying all individual atomic forces must be zero.
- [ ] **(C)** $\nabla_{\mathbf{r}_1} E = \nabla_{\mathbf{r}_2} E = \dots = \nabla_{\mathbf{r}_m} E$ for every geometry.
- [ ] **(D)** $\sum_{i=1}^m \mathbf{r}_i \cdot \mathbf{F}_i = 0$ for all interatomic distances.

In [6]:
ws.render_section('q04_fd_cost_scaling_numeric')

**Q4 · easy** · 💡

Suppose a simulation box contains $N = 500$ atoms in 3D space. If you tried to compute the full $3N$-component force vector at a single MD timestep using **central finite differences** $F_{i,\alpha} \approx -\frac{E(\mathbf{r}_i + \delta\hat{\mathbf{e}}_\alpha) - E(\mathbf{r}_i - \delta\hat{\mathbf{e}}_\alpha)}{2\delta}$ via full-system energy evaluations, how many total evaluations of $E_{\text{total}}$ would be required at **each** timestep?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Each of the N = 500 atoms has 3 Cartesian coordinates (x, y, z), and central differencing requires 2 energy evaluations (+delta and -delta) per coordinate.

</div>
</details>

_Numeric answer required._

In [7]:
ws.render_section('q05_rotational_invariance_torque')

**Q5 · medium** · 💡

In addition to translational invariance, an internal force field depends only on scalar dot products and cross-product magnitudes that are invariant under rigid 3D rotations $\delta \mathbf{r}_i = \delta\boldsymbol{\omega} \times \mathbf{r}_i$. Which physical conservation law follows directly from this rotational invariance of $E_{\text{total}}$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Write delta E = sum_i (nabla_{r_i} E) . (delta omega x r_i) = -delta omega . sum_i (r_i x F_i) = 0 for any infinitesimal rotation axis delta omega.

</div>
</details>

- [ ] **(A)** Zero net internal torque: $\sum_i (\mathbf{r}_i \times \mathbf{F}_i) = 0$, ensuring exact conservation of total angular momentum for an isolated cluster.
- [ ] **(B)** Zero virial pressure: $\sum_i (\mathbf{r}_i \cdot \mathbf{F}_i) = 0$ at all densities.
- [ ] **(C)** Each individual force vector $\mathbf{F}_i$ must point radially toward the coordinate origin.
- [ ] **(D)** The Hessian matrix of second derivatives must be diagonal.

In [8]:
ws.render_section('sec_two_body_bond_stretching')

## 2. Two-Body Radial Potentials & Bond-Stretching Derivatives

Both the covalent **bond-stretching** energy $E_{\text{bond}}(r) = K_r(r - r_{\text{eq}})^2$ and the **non-bonded** pair interactions (Coulomb + Lennard-Jones) depend exclusively on the scalar distance $r = r_{12} = |\mathbf{r}_1 - \mathbf{r}_2|$ between two atoms at positions $\mathbf{r}_1$ and $\mathbf{r}_2$.

### Step 1: Relative Vectors and Sign Conventions
Following the lecture notes (`Interatomic-Derivatives.pdf`, p. 1), define the relative displacement vectors:
$$\mathbf{r}_{12} = \mathbf{r}_1 - \mathbf{r}_2 \quad (\text{points from atom } 2 \to \text{atom } 1), \qquad \mathbf{r}_{21} = \mathbf{r}_2 - \mathbf{r}_1 = -\mathbf{r}_{12} \quad (\text{points from atom } 1 \to \text{atom } 2)$$
with scalar distance:
$$r = r_{12} = r_{21} = |\mathbf{r}_1 - \mathbf{r}_2| = \sqrt{(x_1 - x_2)^2 + (y_1 - y_2)^2 + (z_1 - z_2)^2}$$
and corresponding unit vectors:
$$\hat{\mathbf{r}}_{12} = \frac{\mathbf{r}_1 - \mathbf{r}_2}{|\mathbf{r}_1 - \mathbf{r}_2|} = \frac{\mathbf{r}_{12}}{r_{12}}, \qquad \hat{\mathbf{r}}_{21} = \frac{\mathbf{r}_2 - \mathbf{r}_1}{|\mathbf{r}_2 - \mathbf{r}_1|} = \frac{\mathbf{r}_{21}}{r_{21}} = -\hat{\mathbf{r}}_{12}$$

### Step 2: Gradient of the Interatomic Distance
Differentiating $r_{12} = \bigl[(\mathbf{r}_1 - \mathbf{r}_2)\cdot(\mathbf{r}_1 - \mathbf{r}_2)\bigr]^{1/2}$ with respect to $\mathbf{r}_1$ gives:
$$\frac{\partial r_{12}}{\partial x_1} = \frac{1}{2 r_{12}} \cdot 2(x_1 - x_2) = \frac{x_1 - x_2}{r_{12}} \implies \nabla_{\mathbf{r}_1} |\mathbf{r}_1 - \mathbf{r}_2| = \frac{\mathbf{r}_1 - \mathbf{r}_2}{|\mathbf{r}_1 - \mathbf{r}_2|} = \frac{\mathbf{r}_{12}}{r_{12}} = \hat{\mathbf{r}}_{12} = -\hat{\mathbf{r}}_{21}$$
*(Note on p. 1 of the handout: the handout writes $\nabla_{\mathbf{r}_1}$ and $\nabla_{\mathbf{r}_2}$ with opposite sign definitions when identifying the unit vector pointing from $1\to 2$ vs. $2\to 1$; always verify the physical direction! Increasing $\mathbf{r}_1$ in the direction away from $\mathbf{r}_2$, i.e. along $+\hat{\mathbf{r}}_{12}$, increases the distance $r_{12}$, so $\nabla_{\mathbf{r}_1} r_{12} = +\hat{\mathbf{r}}_{12}$ and $\nabla_{\mathbf{r}_2} r_{12} = -\hat{\mathbf{r}}_{12} = +\hat{\mathbf{r}}_{21}$.)*

### Step 3: Chain Rule for Any Pair Potential $v(r)$
For any pair potential $v(r)$, the force on atom $1$ is:
$$\mathbf{F}_1 = -\nabla_{\mathbf{r}_1} v(r) = -\left.\frac{dv}{dr}\right|_{r=r_{12}} \nabla_{\mathbf{r}_1}|\mathbf{r}_1 - \mathbf{r}_2| = -\frac{dv}{dr}\,\hat{\mathbf{r}}_{12} = -\frac{1}{r_{12}}\frac{dv}{dr}(\mathbf{r}_1 - \mathbf{r}_2)$$
and by Newton's Third Law (antisymmetry $\hat{\mathbf{r}}_{21} = -\hat{\mathbf{r}}_{12}$):
$$\mathbf{F}_2 = -\nabla_{\mathbf{r}_2} v(r) = -\frac{dv}{dr}\,\hat{\mathbf{r}}_{21} = +\frac{dv}{dr}\,\hat{\mathbf{r}}_{12} = -\mathbf{F}_1$$

- **Harmonic Bond Stretch** $v(r) = K_r(r - r_{\text{eq}})^2$:
  $$\frac{dv}{dr} = 2K_r(r_{12} - r_{\text{eq}}) \implies \mathbf{F}_1 = -2K_r(r_{12} - r_{\text{eq}})\frac{\mathbf{r}_1 - \mathbf{r}_2}{r_{12}}, \quad \mathbf{F}_2 = -\mathbf{F}_1$$
- **Lennard-Jones + Coulomb Non-Bonded Pair**:
  $$\frac{dv_{\text{nb}}}{dr} = -\frac{q_i q_j e^2}{r_{ij}^2} + \frac{4\epsilon_{ij}}{r_{ij}}\left[-12\left(\frac{\sigma_{ij}}{r_{ij}}\right)^{12} + 6\left(\frac{\sigma_{ij}}{r_{ij}}\right)^6\right]$$


In [9]:
ws.render_section('concept_bond_stretch_explorer')

### Interactive Explorer: 2-Body Bond Stretch Potential & Force Vectors

Adjust the interatomic separation $r_{12}$, equilibrium bond length $r_0$, and bond stiffness $K_r$, and toggle between the energy/force curves and the 2-atom vector diagram. Observe how $\mathbf{F}_1$ reverses sign across $r_{12} = r_0$ while $\mathbf{F}_1 + \mathbf{F}_2 = \mathbf{0}$ holds everywhere.


**Key Takeaway:** When the bond is stretched ($r_{12} > r_0$), $\frac{dE}{dr_{12}} > 0$, so $\mathbf{F}_1 = -\frac{dE}{dr_{12}}\hat{\mathbf{r}}_{12}$ points in the $-\hat{\mathbf{r}}_{12} = +\hat{\mathbf{r}}_{21}$ direction (pulling atom $1$ toward atom $2$). In code, writing $\mathbf{F}_1 = -\left(\frac{1}{r_{12}}\frac{dv}{dr}\right)\mathbf{r}_{12}$ combines the unit vector normalization with the scalar derivative.


In [10]:
ws.render_section('q06_distance_gradient_identity')

**Q6 · easy** · 💡

Let $r_{12} = |\mathbf{r}_1 - \mathbf{r}_2| = \sqrt{(x_1-x_2)^2 + (y_1-y_2)^2 + (z_1-z_2)^2}$ and define $\mathbf{r}_{12} = \mathbf{r}_1 - \mathbf{r}_2$. What is the exact gradient $\nabla_{\mathbf{r}_1} r_{12}$ with respect to the coordinates of atom $1$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Differentiate [(r_1 - r_2) . (r_1 - r_2)]^(1/2) with respect to r_1.

</div>
</details>

- [ ] **(A)** $\frac{\mathbf{r}_1 - \mathbf{r}_2}{|\mathbf{r}_1 - \mathbf{r}_2|} = \frac{\mathbf{r}_{12}}{r_{12}} = \hat{\mathbf{r}}_{12}$ (the unit vector pointing from atom 2 to atom 1)
- [ ] **(B)** $2(\mathbf{r}_1 - \mathbf{r}_2)$
- [ ] **(C)** $\frac{\mathbf{r}_1 - \mathbf{r}_2}{|\mathbf{r}_1 - \mathbf{r}_2|^3}$
- [ ] **(D)** $\frac{1}{|\mathbf{r}_1 - \mathbf{r}_2|}$ (a scalar rather than a vector)

In [11]:
ws.render_section('q07_stretched_bond_force_direction')

**Q7 · easy** · 💡

Consider two bonded atoms with equilibrium bond length $r_{\text{eq}} = 1.50\text{ Å}$. At a given MD step, $r_{12} = 1.80\text{ Å}$ ($r_{12} > r_{\text{eq}}$). Using $\mathbf{r}_{12} = \mathbf{r}_1 - \mathbf{r}_2$ and $\mathbf{r}_{21} = \mathbf{r}_2 - \mathbf{r}_1$, in which direction does the restoring force $\mathbf{F}_1$ on atom $1$ point?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

When a spring is stretched beyond its equilibrium length, does the force on atom 1 pull it toward atom 2 or push it away from atom 2?

</div>
</details>

- [ ] **(A)** Along $+\hat{\mathbf{r}}_{21} = -\hat{\mathbf{r}}_{12}$ (pointing from atom 1 toward atom 2)
- [ ] **(B)** Along $+\hat{\mathbf{r}}_{12}$ (pointing from atom 2 away from atom 1)
- [ ] **(C)** Perpendicular to the bond vector $\mathbf{r}_{12}$
- [ ] **(D)** The force is zero because $dv/dr$ vanishes when $r_{12} > r_{\text{eq}}$

In [12]:
ws.render_section('q08_harmonic_bond_force_numeric')

**Q8 · medium** · 💡

Two atoms lie on the $x$-axis with atom $1$ at $\mathbf{r}_1 = (2.1, 0, 0)\text{ Å}$ and atom $2$ at $\mathbf{r}_2 = (0.5, 0, 0)\text{ Å}$. They interact via the OPLS harmonic bond potential $E_{\text{bond}}(r) = K_r(r - r_{\text{eq}})^2$ with $K_r = 12.5\text{ eV/Å}^2$ and $r_{\text{eq}} = 1.2\text{ Å}$. What is the $x$-component of the force on atom $1$, $F_{1,x}$ (in $\text{eV/Å}$)? *(Include the correct algebraic sign.)*


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

First compute r_12 = 2.1 - 0.5 = 1.6 A. Then evaluate F_{1,x} = -2 * K_r * (r_12 - r_eq) * (x_1 - x_2) / r_12.

</div>
</details>

_Numeric answer required._

In [13]:
ws.render_section('q09_3d_bond_force_component_numeric')

**Q9 · medium** · 💡

In 3D space, atom $1$ is at $\mathbf{r}_1 = (3.0, 4.0, 0.0)\text{ Å}$ and atom $2$ is at the origin $\mathbf{r}_2 = (0.0, 0.0, 0.0)\text{ Å}$, so $r_{12} = 5.0\text{ Å}$. If $E_{\text{bond}}(r) = K_r(r - r_{\text{eq}})^2$ with $K_r = 10.0\text{ eV/Å}^2$ and $r_{\text{eq}} = 4.0\text{ Å}$, what is the $y$-component of the force on atom $2$, $F_{2,y}$ (in $\text{eV/Å}$)?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

First find dE/dr = 2 * 10.0 * (5.0 - 4.0) = 20.0 eV/Å. Since F_2 = -F_1 = +(dE/dr) * (r_1 - r_2)/r_12, multiply 20.0 by (y_1 - y_2)/r_12 = 4.0/5.0.

</div>
</details>

_Numeric answer required._

In [14]:
ws.render_section('q10_lj_pair_force_form')

**Q10 · medium** · 💡

In MD codes, why do programmers express radial pair forces as $\mathbf{F}_1 = -\left(\frac{1}{r_{12}}\frac{dv}{dr}\right)\mathbf{r}_{12}$ for the Lennard-Jones potential $v_{\text{LJ}}(r) = 4\epsilon\left[\left(\frac{\sigma}{r}\right)^{12} - \left(\frac{\sigma}{r}\right)^6\right]$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compute (1/r) * (dv_LJ / dr) and notice that all powers of r become even powers (r^-2, r^-8, r^-14) expressible purely in terms of r^2 = r_12 . r_12 without taking a square root!

</div>
</details>

- [ ] **(A)** Because $-\frac{1}{r}\frac{dv_{\text{LJ}}}{dr} = \frac{24\epsilon}{r^2} \left[2\left(\frac{\sigma^2}{r^2}\right)^6 - \left(\frac{\sigma^2}{r^2}\right)^3\right]$ depends only on even powers of $r$ (i.e. $r^2 = |\mathbf{r}_{12}|^2$), completely avoiding expensive square-root calls!
- [ ] **(B)** Because the Lennard-Jones potential has no repulsive core when divided by $r_{12}$.
- [ ] **(C)** Because dividing by $r_{12}$ converts the 2-body potential into a 3-body potential.
- [ ] **(D)** Because $dv_{\text{LJ}}/dr$ is constant for all $r_{12}$.

In [15]:
ws.render_section('q11_coulomb_lj_gradient')

**Q11 · medium** · 💡

For two non-bonded atoms $i$ and $j$ with partial charges $q_i, q_j$ separated by $\mathbf{r}_{ij} = \mathbf{r}_i - \mathbf{r}_j$, the electrostatic energy is $v_{\text{Coul}}(r_{ij}) = \frac{q_i q_j e^2}{r_{ij}}$. What is the exact electrostatic force $\mathbf{F}_i^{\text{Coul}} = -\nabla_{\mathbf{r}_i} v_{\text{Coul}}(r_{ij})$ on atom $i$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Differentiate v_Coul(r) = q_i q_j e^2 / r with respect to r to get dv/dr = -q_i q_j e^2 / r^2, then apply F_i = -(dv/dr) * (r_ij / r_ij).

</div>
</details>

- [ ] **(A)** $\mathbf{F}_i^{\text{Coul}} = +\frac{q_i q_j e^2}{r_{ij}^3}(\mathbf{r}_i - \mathbf{r}_j)$
- [ ] **(B)** $\mathbf{F}_i^{\text{Coul}} = -\frac{q_i q_j e^2}{r_{ij}^3}(\mathbf{r}_i - \mathbf{r}_j)$
- [ ] **(C)** $\mathbf{F}_i^{\text{Coul}} = +\frac{q_i q_j e^2}{r_{ij}^2}(\mathbf{r}_i - \mathbf{r}_j)$
- [ ] **(D)** $\mathbf{F}_i^{\text{Coul}} = -\frac{q_i q_j e^2}{r_{ij}}(\mathbf{r}_i - \mathbf{r}_j)$

In [16]:
ws.render_section('q12_hessian_spring_constant')

**Q12 · medium** · 💡

In the OPLS convention $E_{\text{bond}}(r) = K_r(r - r_{\text{eq}})^2$ (without a factor of $\frac{1}{2}$ in front of $K_r$), what is the second derivative $\frac{d^2 E_{\text{bond}}}{dr^2}$ (the effective Hookean spring constant $k_{\text{eff}}$ in $\text{eV/Å}^2$) when $K_r = 18.5\text{ eV/Å}^2$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Differentiate E_bond(r) = K_r * (r - r_eq)^2 twice with respect to r.

</div>
</details>

_Numeric answer required._

In [17]:
ws.render_section('sec_three_body_bond_bending')

## 3. Three-Body Bond-Bending (Angle) Derivatives

Consider three bonded atoms $1\text{–}2\text{–}3$ where **atom $2$ is the central vertex** bonded to terminal atoms $1$ and $3$. Following `Interatomic-Derivatives.pdf` (p. 2), define the two bond vectors pointing away from the central atom $2$:
$$\mathbf{r}_{12} = \mathbf{r}_1 - \mathbf{r}_2, \qquad \mathbf{r}_{32} = \mathbf{r}_3 - \mathbf{r}_2$$
with bond lengths $r_{12} = |\mathbf{r}_{12}|$, $r_{32} = |\mathbf{r}_{32}|$ and unit vectors $\hat{\mathbf{r}}_{12} = \frac{\mathbf{r}_{12}}{r_{12}}$, $\hat{\mathbf{r}}_{32} = \frac{\mathbf{r}_{32}}{r_{32}}$. The interior bond angle $\theta_{123} \in (0, \pi)$ satisfies:
$$\cos\theta_{123} = \frac{\mathbf{r}_{12}\cdot\mathbf{r}_{32}}{r_{12} r_{32}} = \hat{\mathbf{r}}_{12}\cdot\hat{\mathbf{r}}_{32}$$

### Step 1: Differentiating $\arccos(x)$
Let $x = \cos\theta_{123}$. From single-variable calculus, the derivative of $\theta = \arccos(x)$ is:
$$\frac{d}{dx}\arccos(x) = -\frac{1}{\sqrt{1 - x^2}} = -\frac{1}{\sin\theta_{123}}$$
Therefore, for each atom $i \in \{1, 2, 3\}$, the multivariable chain rule gives:
$$\nabla_{\mathbf{r}_i}\theta_{123} = \nabla_{\mathbf{r}_i}\arccos\left(\frac{\mathbf{r}_{12}\cdot\mathbf{r}_{32}}{r_{12}r_{32}}\right) = -\frac{1}{\sin\theta_{123}}\nabla_{\mathbf{r}_i}\left(\frac{\mathbf{r}_{12}\cdot\mathbf{r}_{32}}{r_{12}r_{32}}\right)$$

### Step 2: Quotient Rule for Terminal Atom $1$
When differentiating with respect to $\mathbf{r}_1$, notice that $\mathbf{r}_{32} = \mathbf{r}_3 - \mathbf{r}_2$ and $r_{32} = |\mathbf{r}_{32}|$ are **constant** with respect to $\mathbf{r}_1$! Using $\nabla_{\mathbf{r}_1}(\mathbf{r}_{12}\cdot\mathbf{r}_{32}) = \mathbf{r}_{32}$ and $\nabla_{\mathbf{r}_1}\left(\frac{1}{r_{12}}\right) = -\frac{1}{r_{12}^2}\nabla_{\mathbf{r}_1}r_{12} = -\frac{\mathbf{r}_{12}}{r_{12}^3}$, the product/quotient rule yields:
$$\nabla_{\mathbf{r}_1}\left(\frac{\mathbf{r}_{12}\cdot\mathbf{r}_{32}}{r_{12}r_{32}}\right) = \frac{1}{r_{12}}\frac{\mathbf{r}_{32}}{r_{32}} - \frac{1}{r_{12}}\left(\frac{\mathbf{r}_{12}\cdot\mathbf{r}_{32}}{r_{12}r_{32}}\right)\frac{\mathbf{r}_{12}}{r_{12}} = \frac{1}{r_{12}}\bigl(\hat{\mathbf{r}}_{32} - \cos\theta_{123}\hat{\mathbf{r}}_{12}\bigr)$$
Multiplying by $-\frac{1}{\sin\theta_{123}}$ gives the exact gradient of $\theta_{123}$ with respect to $\mathbf{r}_1$:
$$\boxed{\nabla_{\mathbf{r}_1}\theta_{123} = -\frac{1}{r_{12}\sin\theta_{123}}\bigl(\hat{\mathbf{r}}_{32} - \cos\theta_{123}\hat{\mathbf{r}}_{12}\bigr)}$$

### Step 3: Symmetry for Atom $3$ and Translational Invariance for Vertex Atom $2$
By interchanging indices $1 \leftrightarrow 3$:
$$\boxed{\nabla_{\mathbf{r}_3}\theta_{123} = -\frac{1}{r_{32}\sin\theta_{123}}\bigl(\hat{\mathbf{r}}_{12} - \cos\theta_{123}\hat{\mathbf{r}}_{32}\bigr)}$$
And because $\theta_{123}$ is translationally invariant ($\sum_{i=1}^3 \nabla_{\mathbf{r}_i}\theta_{123} = \mathbf{0}$), the central vertex gradient is obtained immediately without further differentiation:
$$\boxed{\nabla_{\mathbf{r}_2}\theta_{123} = -\nabla_{\mathbf{r}_1}\theta_{123} - \nabla_{\mathbf{r}_3}\theta_{123}}$$
Finally, for $E_{\text{angle}}(\theta_{123}) = K_\theta(\theta_{123} - \theta_{\text{eq}})^2$, the atomic forces are $\mathbf{F}_i = -\frac{dE_{\text{angle}}}{d\theta_{123}}\nabla_{\mathbf{r}_i}\theta_{123} = -2K_\theta(\theta_{123} - \theta_{\text{eq}})\nabla_{\mathbf{r}_i}\theta_{123}$.

### Crucial Geometric Insight: Gram–Schmidt Orthogonality
Look closely at the vector $\mathbf{u}_1 = \hat{\mathbf{r}}_{32} - \cos\theta_{123}\hat{\mathbf{r}}_{12} = \hat{\mathbf{r}}_{32} - (\hat{\mathbf{r}}_{32}\cdot\hat{\mathbf{r}}_{12})\hat{\mathbf{r}}_{12}$.
- It is the **Gram–Schmidt projection** of $\hat{\mathbf{r}}_{32}$ perpendicular to $\hat{\mathbf{r}}_{12}$ in the $(1\text{–}2\text{–}3)$ plane:
  $$\mathbf{u}_1 \cdot \hat{\mathbf{r}}_{12} = \hat{\mathbf{r}}_{32}\cdot\hat{\mathbf{r}}_{12} - \cos\theta_{123}(\hat{\mathbf{r}}_{12}\cdot\hat{\mathbf{r}}_{12}) = \cos\theta_{123} - \cos\theta_{123} = 0$$
- Its magnitude is $|\mathbf{u}_1| = \sqrt{|\hat{\mathbf{r}}_{32}|^2 - 2\cos^2\theta_{123} + \cos^2\theta_{123}} = \sqrt{1 - \cos^2\theta_{123}} = \sin\theta_{123}$, which cancels the $\sin\theta_{123}$ in the denominator so that $|\nabla_{\mathbf{r}_1}\theta_{123}| = \frac{1}{r_{12}}$ (just as $d\theta = ds / r$!).
- Consequently, **$\mathbf{F}_1$ is strictly perpendicular to bond $\mathbf{r}_{12}$**, and **$\mathbf{F}_3$ is strictly perpendicular to bond $\mathbf{r}_{32}$**! Pure angle bending never stretches or compresses the bonds at first order.


In [18]:
ws.render_section('concept_three_body_angle_forces')

### Interactive Explorer: 3-Body Bond-Bending Force Geometry

Vary the instantaneous bond angle $\theta_{123}$, equilibrium angle $\theta_0$, bond-length ratio $r_{12}/r_{32}$, and bending stiffness $K_\theta$. Verify visually and numerically in the caption that $\mathbf{F}_1 \perp \mathbf{r}_{12}$, $\mathbf{F}_3 \perp \mathbf{r}_{32}$, and $\mathbf{F}_2 = -(\mathbf{F}_1 + \mathbf{F}_3)$.


**Key Takeaway:** Because $|\nabla_{\mathbf{r}_1}\theta_{123}| = \frac{1}{r_{12}}$ and $|\nabla_{\mathbf{r}_3}\theta_{123}| = \frac{1}{r_{32}}$, a longer lever arm ($r_{12} > r_{32}$) requires a **smaller** force $|\mathbf{F}_1| < |\mathbf{F}_3|$ to exert the same bending torque around vertex atom $2$, ensuring zero net torque $\sum_i \mathbf{r}_i \times \mathbf{F}_i = \mathbf{0}$.


In [19]:
ws.render_section('q13_arccos_derivative_chain_rule')

**Q13 · easy** · 💡

When deriving $\nabla_{\mathbf{r}_1}\theta_{123}$ from $\theta_{123} = \arccos(\cos\theta_{123})$ for $\theta_{123} \in (0, \pi)$, what is the derivative $\frac{d}{dx}\arccos(x)$ evaluated at $x = \cos\theta_{123}$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall d/dx arccos(x) = -1 / sqrt(1 - x^2) and substitute x = cos(theta_123).

</div>
</details>

- [ ] **(A)** $-\frac{1}{\sin\theta_{123}}$
- [ ] **(B)** $+\frac{1}{\sin\theta_{123}}$
- [ ] **(C)** $-\sin\theta_{123}$
- [ ] **(D)** $-\frac{1}{\cos\theta_{123}}$

In [20]:
ws.render_section('q14_grad_cos_theta_expression')

**Q14 · medium** · 💡

Using $\cos\theta_{123} = \frac{\mathbf{r}_{12}\cdot\mathbf{r}_{32}}{r_{12}r_{32}}$ with $\mathbf{r}_{12} = \mathbf{r}_1 - \mathbf{r}_2$ and $\mathbf{r}_{32} = \mathbf{r}_3 - \mathbf{r}_2$, what is $\nabla_{\mathbf{r}_1}(\cos\theta_{123})$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Apply the product rule to (1/r_12) * (r_12 . r_32 / r_32), treating r_32 as constant with respect to r_1.

</div>
</details>

- [ ] **(A)** $\frac{1}{r_{12}}(\hat{\mathbf{r}}_{32} - \cos\theta_{123}\,\hat{\mathbf{r}}_{12})$
- [ ] **(B)** $\frac{1}{r_{32}}(\hat{\mathbf{r}}_{12} - \cos\theta_{123}\,\hat{\mathbf{r}}_{32})$
- [ ] **(C)** $-\frac{1}{r_{12}\sin\theta_{123}}(\hat{\mathbf{r}}_{32} - \cos\theta_{123}\,\hat{\mathbf{r}}_{12})$
- [ ] **(D)** $\frac{\hat{\mathbf{r}}_{32} + \hat{\mathbf{r}}_{12}}{r_{12} r_{32}}$

In [21]:
ws.render_section('q15_orthogonality_of_angle_force')

**Q15 · easy** · 💡

For any bond angle $\theta_{123} \in (0, \pi)$ and any bond lengths $r_{12}, r_{32} > 0$, what is the exact value of the scalar dot product $\bigl(\nabla_{\mathbf{r}_1}\theta_{123}\bigr) \cdot \mathbf{r}_{12}$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compute (r_hat_32 - cos(theta_123) * r_hat_12) . r_hat_12 using r_hat_32 . r_hat_12 = cos(theta_123) and r_hat_12 . r_hat_12 = 1.

</div>
</details>

_Numeric answer required._

In [22]:
ws.render_section('q16_magnitude_of_inplane_projection_vector')

**Q16 · medium** · 💡

In the angle gradient formula $\nabla_{\mathbf{r}_1}\theta_{123} = -\frac{1}{r_{12}\sin\theta_{123}}(\hat{\mathbf{r}}_{32} - \cos\theta_{123}\hat{\mathbf{r}}_{12})$, what is the Euclidean norm (magnitude) of the vector $\mathbf{u}_1 = \hat{\mathbf{r}}_{32} - \cos\theta_{123}\hat{\mathbf{r}}_{12}$, and what does this imply for $|\nabla_{\mathbf{r}_1}\theta_{123}|$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compute |u_1|^2 = (r_hat_32 - cos(theta)*r_hat_12) . (r_hat_32 - cos(theta)*r_hat_12).

</div>
</details>

- [ ] **(A)** $|\mathbf{u}_1| = \sin\theta_{123}$, which cancels the $\sin\theta_{123}$ in the denominator so that $|\nabla_{\mathbf{r}_1}\theta_{123}| = \frac{1}{r_{12}}$.
- [ ] **(B)** $|\mathbf{u}_1| = \cos\theta_{123}$, so $|\nabla_{\mathbf{r}_1}\theta_{123}| = \frac{\cot\theta_{123}}{r_{12}}$.
- [ ] **(C)** $|\mathbf{u}_1| = 1$, so $|\nabla_{\mathbf{r}_1}\theta_{123}| = \frac{1}{r_{12}\sin\theta_{123}}$.
- [ ] **(D)** $|\mathbf{u}_1| = r_{12}\sin\theta_{123}$, so $|\nabla_{\mathbf{r}_1}\theta_{123}| = 1$.

In [23]:
ws.render_section('q17_direction_of_grad_r1_theta')

**Q17 · medium** · 💡

Why is there a minus sign in front of $\nabla_{\mathbf{r}_1}\theta_{123} = -\frac{1}{r_{12}\sin\theta_{123}}(\hat{\mathbf{r}}_{32} - \cos\theta_{123}\hat{\mathbf{r}}_{12})$? Specifically, which way does the vector $\nabla_{\mathbf{r}_1}\theta_{123}$ point in the $(1\text{–}2\text{–}3)$ plane?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall that the gradient nabla_{r_1} theta_123 points in the direction of the displacement delta r_1 that INCREASES theta_123.

</div>
</details>

- [ ] **(A)** Because $(\hat{\mathbf{r}}_{32} - \cos\theta_{123}\,\hat{\mathbf{r}}_{12})$ points inward toward bond $\mathbf{r}_{32}$ (which decreases $\theta_{123}$), so multiplying by $-1$ makes $\nabla_{\mathbf{r}_1}\theta_{123}$ point outward away from bond $\mathbf{r}_{32}$ (the direction that increases $\theta_{123}$).
- [ ] **(B)** Because $\nabla_{\mathbf{r}_1}\theta_{123}$ points in the direction that decreases $\theta_{123}$.
- [ ] **(C)** Because $\sin\theta_{123}$ is negative for all acute and obtuse bond angles.
- [ ] **(D)** Because the vector points perpendicular to the (1-2-3) plane.

In [24]:
ws.render_section('q18_right_angle_gradient_numeric')

**Q18 · medium** · 💡

Suppose vertex atom $2$ is at the origin $(0,0,0)$, atom $1$ is on the $+x$ axis at $\mathbf{r}_1 = (2.0, 0, 0)\text{ Å}$, and atom $3$ is on the $+y$ axis at $\mathbf{r}_3 = (0, 4.0, 0)\text{ Å}$, forming a right angle $\theta_{123} = \frac{\pi}{2}\text{ rad}$ ($90^\circ$). What is the $y$-component of $\nabla_{\mathbf{r}_1}\theta_{123}$ (in $\text{rad/Å}$)? *(Include the correct algebraic sign.)*


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

At theta = 90 deg, cos(theta) = 0, sin(theta) = 1, r_12 = 2.0 A, and r_hat_32 = (0, 1, 0). Substitute into nabla_{r_1} theta_123 = -(1 / (r_12 * sin(theta))) * (r_hat_32 - cos(theta)*r_hat_12).

</div>
</details>

_Numeric answer required._

In [25]:
ws.render_section('q19_right_angle_vertex_force_numeric')

**Q19 · hard** · 💡

Using the same $90^\circ$ geometry ($\mathbf{r}_2 = (0,0,0)$, $\mathbf{r}_1 = (2.0, 0, 0)\text{ Å}$, $\mathbf{r}_3 = (0, 4.0, 0)\text{ Å}$), suppose $E_{\text{angle}}(\theta) = K_\theta(\theta - \theta_{\text{eq}})^2$ has derivative $\frac{dE_{\text{angle}}}{d\theta} = 2K_\theta(\theta - \theta_{\text{eq}}) = -6.0\text{ eV/rad}$ at $\theta = \pi/2$. What is the Euclidean magnitude $|\mathbf{F}_2|$ of the force on the central vertex atom $2$ (in $\text{eV/Å}$)?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

First find F_1 = -(dE/dtheta)*nabla_{r_1}theta = (0, -3.0, 0) eV/Å and F_3 = -(dE/dtheta)*nabla_{r_3}theta = (-1.5, 0, 0) eV/Å. Then use F_2 = -(F_1 + F_3) and compute sqrt(F_{2,x}^2 + F_{2,y}^2).

</div>
</details>

_Numeric answer required._

In [26]:
ws.render_section('q20_terminal_force_ratio_numeric')

**Q20 · medium** · 💡

In an asymmetric triatomic molecule $1\text{–}2\text{–}3$ (such as $\text{H–O–Cl}$), suppose bond $1\text{–}2$ has length $r_{12} = 0.96\text{ Å}$ and bond $3\text{–}2$ has length $r_{32} = 1.68\text{ Å}$. For any non-equilibrium bond angle $\theta_{123} \ne \theta_{\text{eq}}$, what is the exact ratio $|\mathbf{F}_1| / |\mathbf{F}_3|$ of the angle-bending force magnitudes on the two terminal atoms?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall |F_1| = |dE/dtheta| / r_12 and |F_3| = |dE/dtheta| / r_32, so |F_1| / |F_3| = r_32 / r_12.

</div>
</details>

_Numeric answer required._

In [27]:
ws.render_section('q21_collinear_angle_singularity')

**Q21 · hard** · 💡

What happens to the analytical angle-gradient formula $\nabla_{\mathbf{r}_1}\theta_{123} = -\frac{1}{r_{12}\sin\theta_{123}}(\hat{\mathbf{r}}_{32} - \cos\theta_{123}\hat{\mathbf{r}}_{12})$ as the three atoms become collinear ($\theta_{123} \to \pi = 180^\circ$ or $\theta_{123} \to 0^\circ$)?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check what happens to both the numerator (r_hat_32 - cos(theta)*r_hat_12) and the denominator sin(theta_123) when theta -> pi (where r_hat_32 = -r_hat_12 and cos(theta) = -1).

</div>
</details>

- [ ] **(A)** Both the numerator vector $(\hat{\mathbf{r}}_{32} - \cos\theta_{123}\,\hat{\mathbf{r}}_{12})$ and the denominator $\sin\theta_{123}$ vanish simultaneously ($0/0$ indeterminacy) because the bending plane is no longer uniquely defined for a collinear triplet.
- [ ] **(B)** The numerator remains unit length while $\sin\theta_{123} \to 1$, so no numerical issue occurs.
- [ ] **(C)** $\cos\theta_{123}$ diverges to infinity at $\theta = 180^\circ$.
- [ ] **(D)** The bond lengths $r_{12}$ and $r_{32}$ vanish at $\theta = 180^\circ$.

In [28]:
ws.render_section('sec_four_body_torsion_blondel_karplus')

## 4. Four-Body Torsion (Dihedral) Potentials & The Blondel–Karplus Formula

A **torsion (dihedral) angle** $\phi$ requires a chain of four bonded atoms $1\text{–}2\text{–}3\text{–}4$ at positions $\mathbf{r}_1, \mathbf{r}_2, \mathbf{r}_3, \mathbf{r}_4$. In OPLS-AA (`Interatomic-Derivatives.pdf`, p. 1), the torsion potential is expanded as a truncated Fourier series:
$$E(\phi) = \sum_{n=1}^{3} \frac{V_n}{2}\bigl(1 + \cos(n\phi + f_n)\bigr)$$
(often written with phase offsets $f_1=0, f_2=\pi, f_3=0$ so that $E(\phi) = \frac{V_1}{2}(1+\cos\phi) + \frac{V_2}{2}(1-\cos 2\phi) + \frac{V_3}{2}(1+\cos 3\phi)$). Differentiating with respect to $\phi$ yields the scalar torsional torque:
$$\frac{dE}{d\phi} = -\sum_{n=1}^{3} \frac{n V_n}{2}\sin(n\phi + f_n)$$

### Step 1: Plane Normal Vectors $\mathbf{n}_{123}$ and $\mathbf{n}_{432}$
The torsion angle $\phi$ is defined as the angle between the **$(1\text{–}2\text{–}3)$ plane** and the **$(2\text{–}3\text{–}4)$ plane**, which intersect along the central bond connecting atoms $2$ and $3$. Using the relative bond vectors $\mathbf{r}_{ij} = \mathbf{r}_i - \mathbf{r}_j$ (`Interatomic-Derivatives.pdf`, p. 2), the (unnormalized) normal vectors to the two planes are:
$$\mathbf{n}_{123} = \mathbf{r}_{12} \times \mathbf{r}_{32}, \qquad \mathbf{n}_{432} = \mathbf{r}_{43} \times \mathbf{r}_{23}$$
with magnitudes $n_{123} = |\mathbf{n}_{123}| = r_{12}r_{32}\sin\theta_{123}$ and $n_{432} = |\mathbf{n}_{432}| = r_{43}r_{23}\sin\theta_{432}$, where $r_{23} = |\mathbf{r}_{23}| = |\mathbf{r}_{32}|$. The torsion angle satisfies:
$$\cos\phi = -\frac{\mathbf{n}_{123}\cdot\mathbf{n}_{432}}{|\mathbf{n}_{123}|\,|\mathbf{n}_{432}|}$$

### Step 2: Why Naive $\arccos(\cos\phi)$ Differentiation Fails Catastrophically
Unlike a 3-body bond angle $\theta_{123}$ (which stays near $\theta_{\text{eq}} \approx 109.5^\circ$ and never crosses $0^\circ$ or $180^\circ$ in normal covalent bonds), a dihedral angle $\phi$ **routinely rotates through $\phi = 0^\circ$ (*cis*/eclipsed) and $\phi = \pm 180^\circ$ (*trans*/anti)**!
- If you naively apply $\nabla_{\mathbf{r}_i}\phi = -\frac{1}{\sin\phi}\nabla_{\mathbf{r}_i}(\cos\phi)$, the factor $\frac{1}{\sin\phi}$ **blows up to $\pm\infty$** at $\phi = 0^\circ$ and $\phi = 180^\circ$ (where all four atoms are coplanar and $\sin\phi = 0$), even though the physical force at a planar configuration is completely smooth and finite!

### Step 3: The Singularity-Free Blondel–Karplus Equations (1996)
Blondel and Karplus (*J. Comp. Chem.* **17**, 1132, 1996; Eq. 27 on p. 3 of `Interatomic-Derivatives.pdf`) proved that by expressing $\nabla_{\mathbf{r}_i}\phi$ directly in terms of the plane normals $\mathbf{n}_{123}$ and $\mathbf{n}_{432}$, the $1/\sin\phi$ singularity cancels analytically! In our handout's notation:
$$\boxed{\nabla_{\mathbf{r}_1}\phi = \frac{r_{23}}{n_{123}^2}\mathbf{n}_{123}}, \qquad \boxed{\nabla_{\mathbf{r}_4}\phi = \frac{r_{23}}{n_{432}^2}\mathbf{n}_{432}}$$
$$\boxed{\nabla_{\mathbf{r}_2}\phi = -\nabla_{\mathbf{r}_1}\phi + \frac{\mathbf{r}_{12}\cdot\mathbf{r}_{32}}{n_{123}^2 r_{23}}\mathbf{n}_{123} - \frac{\mathbf{r}_{43}\cdot\mathbf{r}_{23}}{n_{432}^2 r_{23}}\mathbf{n}_{432}}$$
$$\boxed{\nabla_{\mathbf{r}_3}\phi = -\nabla_{\mathbf{r}_4}\phi - \frac{\mathbf{r}_{12}\cdot\mathbf{r}_{32}}{n_{123}^2 r_{23}}\mathbf{n}_{123} + \frac{\mathbf{r}_{43}\cdot\mathbf{r}_{23}}{n_{432}^2 r_{23}}\mathbf{n}_{432}}$$

### Physical & Geometric Properties of the Blondel–Karplus Gradients
1. **No $1/\sin\phi$ singularity:** $\nabla_{\mathbf{r}_1}\phi$ depends *only* on $r_{23}$ and $\mathbf{n}_{123}$, which remain non-zero for all $\phi \in [-\pi, \pi]$ as long as the 3-body bond angle $\theta_{123} \notin \{0, \pi\}$.
2. **Pure out-of-plane terminal forces:** $\nabla_{\mathbf{r}_1}\phi$ is parallel to $\mathbf{n}_{123}$ (perpendicular to the $1\text{–}2\text{–}3$ plane), with magnitude:
   $$|\nabla_{\mathbf{r}_1}\phi| = \frac{r_{23}}{n_{123}} = \frac{r_{23}}{r_{12}r_{23}\sin\theta_{123}} = \frac{1}{r_{12}\sin\theta_{123}} = \frac{1}{d_1}$$
   where $d_1 = r_{12}\sin\theta_{123}$ is the perpendicular lever-arm distance of atom $1$ from the central rotation axis $\mathbf{r}_{23}$!
3. **Exact Translational Invariance:** Adding the four equations together, the $\frac{\mathbf{r}_{12}\cdot\mathbf{r}_{32}}{n_{123}^2 r_{23}}\mathbf{n}_{123}$ and $\frac{\mathbf{r}_{43}\cdot\mathbf{r}_{23}}{n_{432}^2 r_{23}}\mathbf{n}_{432}$ projection terms cancel in pairs:
   $$\nabla_{\mathbf{r}_1}\phi + \nabla_{\mathbf{r}_2}\phi + \nabla_{\mathbf{r}_3}\phi + \nabla_{\mathbf{r}_4}\phi = \mathbf{0} \implies \mathbf{F}_1 + \mathbf{F}_2 + \mathbf{F}_3 + \mathbf{F}_4 = \mathbf{0}$$


In [29]:
ws.render_section('concept_opls_dihedral_torsion')

### Interactive Explorer: OPLS Dihedral Potential & Blondel–Karplus Forces

Adjust the OPLS Fourier coefficients $V_1, V_2, V_3$ and dihedral angle $\phi_{1234} \in [-180^\circ, 180^\circ]$. Notice how both $E(\phi)$ and the Blondel–Karplus force $\mathbf{F}_1 = -\frac{dE}{d\phi}\nabla_{\mathbf{r}_1}\phi$ remain completely smooth across the planar configurations $\phi = 0^\circ, \pm 180^\circ$ (vertical dotted lines).


**Key Takeaway:** In ethane-like $sp^3\text{–}sp^3$ bonds ($V_1=V_2=0, V_3 > 0$), $E(\phi) = \frac{V_3}{2}(1+\cos 3\phi)$ exhibits three barriers and three *gauche/trans* minima. Because Blondel–Karplus expresses $\nabla_{\mathbf{r}_i}\phi$ directly via $\mathbf{n}_{123}$ and $\mathbf{n}_{432}$, crossing $\phi = 0^\circ$ or $\pm 180^\circ$ never causes a $1/\sin\phi$ division-by-zero crash.


In [30]:
ws.render_section('q22_opls_torsion_derivative_formula')

**Q22 · easy** · 💡

For the OPLS torsion energy $E(\phi) = \sum_{n=1}^{3} \frac{V_n}{2}\bigl(1 + \cos(n\phi + f_n)\bigr)$, what is the analytical derivative $\frac{dE}{d\phi}$ with respect to the torsion angle $\phi$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Differentiate cos(n*phi + f_n) with respect to phi using the chain rule: d/dphi cos(n*phi + f_n) = -n * sin(n*phi + f_n).

</div>
</details>

- [ ] **(A)** $\frac{dE}{d\phi} = -\sum_{n=1}^3 \frac{n V_n}{2} \sin(n\phi + f_n)$
- [ ] **(B)** $\frac{dE}{d\phi} = +\sum_{n=1}^3 \frac{V_n}{2} \sin(n\phi + f_n)$
- [ ] **(C)** $\frac{dE}{d\phi} = -\sum_{n=1}^3 \frac{V_n}{2n} \sin(n\phi + f_n)$
- [ ] **(D)** $\frac{dE}{d\phi} = +\sum_{n=1}^3 \frac{n V_n}{2} \cos(n\phi + f_n)$

In [31]:
ws.render_section('q23_opls_torsion_derivative_numeric')

**Q23 · medium** · 💡

Consider a pure 3-fold dihedral barrier $E(\phi) = \frac{V_3}{2}(1 + \cos 3\phi)$ (i.e. $V_1 = V_2 = 0$, $f_3 = 0$) with $V_3 = 4.0\text{ kcal/mol}$. What is the scalar derivative $\frac{dE}{d\phi}$ (in $\text{kcal/(mol}\cdot\text{rad)}$) at $\phi = 30^\circ = \frac{\pi}{6}\text{ rad}$? *(Include the correct algebraic sign.)*


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate dE/dphi = -(3 * V_3 / 2) * sin(3 * phi) at 3 * phi = 90 deg, where sin(90 deg) = 1.

</div>
</details>

_Numeric answer required._

In [32]:
ws.render_section('q24_plane_normal_definitions')

**Q24 · easy** · 💡

In the 4-atom chain $1\text{–}2\text{–}3\text{–}4$ (`Interatomic-Derivatives.pdf`, p. 2), how are the unnormalized plane normal vectors $\mathbf{n}_{123}$ (for the $1\text{–}2\text{–}3$ plane) and $\mathbf{n}_{432}$ (for the $4\text{–}3\text{–}2$ plane) defined in terms of relative bond vectors $\mathbf{r}_{ij} = \mathbf{r}_i - \mathbf{r}_j$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check the cross products of the two bond vectors sharing vertex 2 (for 1-2-3) and sharing vertex 3 (for 4-3-2) on page 2 of the handout.

</div>
</details>

- [ ] **(A)** $\mathbf{n}_{123} = \mathbf{r}_{12} \times \mathbf{r}_{32}$ and $\mathbf{n}_{432} = \mathbf{r}_{43} \times \mathbf{r}_{23}$
- [ ] **(B)** $\mathbf{n}_{123} = \mathbf{r}_{12} \cdot \mathbf{r}_{32}$ and $\mathbf{n}_{432} = \mathbf{r}_{43} \cdot \mathbf{r}_{23}$
- [ ] **(C)** $\mathbf{n}_{123} = \mathbf{r}_{12} + \mathbf{r}_{32}$ and $\mathbf{n}_{432} = \mathbf{r}_{43} + \mathbf{r}_{23}$
- [ ] **(D)** $\mathbf{n}_{123} = \mathbf{r}_{14} \times \mathbf{r}_{23}$ and $\mathbf{n}_{432} = \mathbf{r}_{12} \times \mathbf{r}_{43}$

In [33]:
ws.render_section('q25_why_arccos_fails_for_torsion')

**Q25 · medium** · 💡

Why does differentiating $\phi = \arccos\left(-\frac{\mathbf{n}_{123}\cdot\mathbf{n}_{432}}{n_{123}n_{432}}\right)$ via $-\frac{1}{\sin\phi}\nabla_{\mathbf{r}_i}(\cos\phi)$ fail in MD simulations of flexible molecules, whereas the Blondel–Karplus formula succeeds?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider what happens when a rotating dihedral angle passes through cis (phi = 0) or trans (phi = 180 deg) planar conformations.

</div>
</details>

- [ ] **(A)** Dihedral angles frequently pass through planar cis ($\phi = 0$) and trans ($\phi = \pi$) conformations where $\sin\phi = 0$, causing $-1/\sin\phi$ to divide by zero; Blondel-Karplus cancels $\sin\phi$ analytically by expressing gradients directly via $\mathbf{n}_{123}$ and $\mathbf{n}_{432}$.
- [ ] **(B)** The arccos formula only works when all four atoms lie in the same plane.
- [ ] **(C)** Blondel-Karplus approximates the torsion force using finite differences instead of exact derivatives.
- [ ] **(D)** The plane normals $\mathbf{n}_{123}$ and $\mathbf{n}_{432}$ vanish at $\phi = 0$ and $\phi = \pi$.

In [34]:
ws.render_section('q26_blondel_karplus_terminal_gradient')

**Q26 · medium** · 💡

According to Equation 27 of Blondel and Karplus (`Interatomic-Derivatives.pdf`, p. 3), what are the exact analytical torsion gradients $\nabla_{\mathbf{r}_1}\phi$ and $\nabla_{\mathbf{r}_4}\phi$ on the two terminal atoms $1$ and $4$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Look at the first two lines on page 3 of Interatomic-Derivatives.pdf.

</div>
</details>

- [ ] **(A)** $\nabla_{\mathbf{r}_1}\phi = \frac{r_{23}}{n_{123}^2}\mathbf{n}_{123}$ and $\nabla_{\mathbf{r}_4}\phi = \frac{r_{23}}{n_{432}^2}\mathbf{n}_{432}$
- [ ] **(B)** $\nabla_{\mathbf{r}_1}\phi = \frac{n_{123}}{r_{23}^2}\mathbf{r}_{23}$ and $\nabla_{\mathbf{r}_4}\phi = \frac{n_{432}}{r_{23}^2}\mathbf{r}_{23}$
- [ ] **(C)** $\nabla_{\mathbf{r}_1}\phi = \frac{1}{n_{123}}\mathbf{r}_{12}$ and $\nabla_{\mathbf{r}_4}\phi = \frac{1}{n_{432}}\mathbf{r}_{43}$
- [ ] **(D)** $\nabla_{\mathbf{r}_1}\phi = -\frac{1}{\sin\phi}\mathbf{n}_{123}$ and $\nabla_{\mathbf{r}_4}\phi = -\frac{1}{\sin\phi}\mathbf{n}_{432}$

In [35]:
ws.render_section('q27_terminal_torsion_force_lever_arm')

**Q27 · medium** · 💡

Since $n_{123} = |\mathbf{r}_{12} \times \mathbf{r}_{32}| = r_{12}r_{23}\sin\theta_{123}$, what does the Euclidean magnitude $|\nabla_{\mathbf{r}_1}\phi| = \left|\frac{r_{23}}{n_{123}^2}\mathbf{n}_{123}\right| = \frac{r_{23}}{n_{123}}$ simplify to, and what is its physical meaning?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Substitute n_123 = r_12 * r_23 * sin(theta_123) into r_23 / n_123 and cancel r_23.

</div>
</details>

- [ ] **(A)** $\frac{1}{r_{12}\sin\theta_{123}}$, which is the reciprocal of the perpendicular distance (lever arm) $d_1$ from atom 1 to the central bond axis 2-3.
- [ ] **(B)** $\frac{r_{23}}{r_{12}^2}$, which depends only on the bond lengths and not on $\theta_{123}$.
- [ ] **(C)** $\frac{\sin\theta_{123}}{r_{12}}$, which vanishes at $\theta_{123} = 90^\circ$.
- [ ] **(D)** $\frac{1}{r_{23}\sin\phi}$, which diverges at $\phi = 0$.

In [36]:
ws.render_section('q28_blondel_karplus_terminal_gradient_numeric')

**Q28 · medium** · 💡

Suppose in a 4-atom chain $1\text{–}2\text{–}3\text{–}4$, the bond lengths are $r_{12} = 1.5\text{ Å}$ and $r_{23} = 2.0\text{ Å}$, and the bond angle $\theta_{123} = 90^\circ$ ($\sin\theta_{123} = 1$, so $n_{123} = r_{12}r_{23}\sin\theta_{123} = 3.0\text{ Å}^2$). If $\frac{dE}{d\phi} = -4.5\text{ eV/rad}$, what is the magnitude $|\mathbf{F}_1| = \left|-\frac{dE}{d\phi}\nabla_{\mathbf{r}_1}\phi\right|$ of the torsion force on atom $1$ (in $\text{eV/Å}$)?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Use |nabla_{r_1} phi| = r_23 / n_123 = 1 / (r_12 * sin(theta_123)) = 1 / 1.5 A^-1, then multiply by |dE/dphi| = 4.5 eV/rad.

</div>
</details>

_Numeric answer required._

In [37]:
ws.render_section('q29_blondel_karplus_internal_atoms_sum')

**Q29 · medium** · 💡

Inspect the Blondel–Karplus expressions for the two central atoms $2$ and $3$ (`Interatomic-Derivatives.pdf`, p. 3):
$$\nabla_{\mathbf{r}_2}\phi = -\nabla_{\mathbf{r}_1}\phi + \frac{\mathbf{r}_{12}\cdot\mathbf{r}_{32}}{n_{123}^2 r_{23}}\mathbf{n}_{123} - \frac{\mathbf{r}_{43}\cdot\mathbf{r}_{23}}{n_{432}^2 r_{23}}\mathbf{n}_{432}$$
$$\nabla_{\mathbf{r}_3}\phi = -\nabla_{\mathbf{r}_4}\phi - \frac{\mathbf{r}_{12}\cdot\mathbf{r}_{32}}{n_{123}^2 r_{23}}\mathbf{n}_{123} + \frac{\mathbf{r}_{43}\cdot\mathbf{r}_{23}}{n_{432}^2 r_{23}}\mathbf{n}_{432}$$
Which vector space do all four gradients $\nabla_{\mathbf{r}_1}\phi, \nabla_{\mathbf{r}_2}\phi, \nabla_{\mathbf{r}_3}\phi, \nabla_{\mathbf{r}_4}\phi$ lie in, and what is their vector sum $\sum_{i=1}^4 \nabla_{\mathbf{r}_i}\phi$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Notice which vectors appear on the right-hand side of all four Blondel-Karplus formulas (only n_123 and n_432!), and add the four equations together.

</div>
</details>

- [ ] **(A)** Every gradient is a linear combination solely of $\mathbf{n}_{123}$ and $\mathbf{n}_{432}$ (hence strictly perpendicular to the central bond vector $\mathbf{r}_{23}$!), and their sum is identically $0$.
- [ ] **(B)** The central atom gradients lie parallel to the central bond $\mathbf{r}_{23}$, and their sum equals $\mathbf{r}_{23}$.
- [ ] **(C)** Only $\nabla_{\mathbf{r}_1}\phi$ and $\nabla_{\mathbf{r}_4}\phi$ are perpendicular to $\mathbf{r}_{23}$, while $\nabla_{\mathbf{r}_2}\phi$ has a non-zero component along $\mathbf{r}_{23}$.
- [ ] **(D)** The sum of the four gradients equals $\frac{\mathbf{n}_{123} + \mathbf{n}_{432}}{r_{23}}$.

In [38]:
ws.render_section('q30_special_case_orthogonal_central_bonds')

**Q30 · hard** · 💡

In the special geometric case where both interior bond angles are right angles ($\theta_{123} = 90^\circ$ and $\theta_{432} = 90^\circ$, so that $\mathbf{r}_{12}\cdot\mathbf{r}_{32} = 0$ and $\mathbf{r}_{43}\cdot\mathbf{r}_{23} = 0$), what do the Blondel–Karplus internal gradients $\nabla_{\mathbf{r}_2}\phi$ and $\nabla_{\mathbf{r}_3}\phi$ simplify to?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Set r_12 . r_32 = 0 and r_43 . r_23 = 0 in the equations for nabla_{r_2} phi and nabla_{r_3} phi on page 3.

</div>
</details>

- [ ] **(A)** $\nabla_{\mathbf{r}_2}\phi = -\nabla_{\mathbf{r}_1}\phi$ and $\nabla_{\mathbf{r}_3}\phi = -\nabla_{\mathbf{r}_4}\phi$ (so $\mathbf{F}_2 = -\mathbf{F}_1$ and $\mathbf{F}_3 = -\mathbf{F}_4$)
- [ ] **(B)** $\nabla_{\mathbf{r}_2}\phi = 0$ and $\nabla_{\mathbf{r}_3}\phi = 0$
- [ ] **(C)** $\nabla_{\mathbf{r}_2}\phi = +\nabla_{\mathbf{r}_1}\phi$ and $\nabla_{\mathbf{r}_3}\phi = +\nabla_{\mathbf{r}_4}\phi$
- [ ] **(D)** $\nabla_{\mathbf{r}_2}\phi = -\nabla_{\mathbf{r}_4}\phi$ and $\nabla_{\mathbf{r}_3}\phi = -\nabla_{\mathbf{r}_1}\phi$

In [39]:
ws.render_section('sec_finite_difference_gradient_testing')

## 5. Debugging Force Fields: Finite-Difference Gradient Verification

As we saw in Sections 2–4, analytical force derivatives for 3-body angles and 4-body dihedrals involve intricate vector cross products, dot products, and index sign conventions ($\mathbf{r}_{12}$ vs. $\mathbf{r}_{21}$). A single sign error in one term will break energy conservation in MD!

### How Every MD Developer Verifies Analytical Forces
Before running a single MD step, you should **always** verify your analytical force routine $\mathbf{F}_i^{\text{analytic}} = -\nabla_{\mathbf{r}_i}E$ against a **numerical finite-difference derivative** on a random, non-symmetric test configuration $\{\mathbf{r}_i\}$:
- **Forward Difference ($\mathcal{O}(\delta)$ truncation error):**
  $$F_{i,\alpha}^{\text{fwd}}(\delta) = -\frac{E(\mathbf{r}_i + \delta\hat{\mathbf{e}}_\alpha) - E(\mathbf{r}_i)}{\delta} = F_{i,\alpha}^{\text{exact}} - \frac{\delta}{2}\frac{\partial^2 E}{\partial r_{i,\alpha}^2} + \mathcal{O}(\delta^2)$$
- **Central Difference ($\mathcal{O}(\delta^2)$ truncation error):**
  $$F_{i,\alpha}^{\text{cen}}(\delta) = -\frac{E(\mathbf{r}_i + \delta\hat{\mathbf{e}}_\alpha) - E(\mathbf{r}_i - \delta\hat{\mathbf{e}}_\alpha)}{2\delta} = F_{i,\alpha}^{\text{exact}} - \frac{\delta^2}{6}\frac{\partial^3 E}{\partial r_{i,\alpha}^3} + \mathcal{O}(\delta^4)$$

### The V-Shaped Error Curve: Truncation vs. Floating-Point Roundoff
On a log-log plot of relative error $\varepsilon(\delta) = \frac{|F^{\text{FD}}(\delta) - F^{\text{analytic}}|}{|F^{\text{analytic}}|}$ versus step size $\delta$:
1. **Large $\delta$ ($\delta \gtrsim 10^{-4}\text{ Å}$): Taylor Truncation Regime.** The error decreases as $\delta^1$ (slope $+1$) for forward difference and as $\delta^2$ (slope $+2$) for central difference.
2. **Tiny $\delta$ ($\delta \lesssim 10^{-7}\text{ Å}$): Subtractive Cancellation Regime.** In IEEE-754 64-bit floating-point arithmetic ($\epsilon_{\text{mach}} \approx 2.22 \times 10^{-16}$), subtracting two nearly identical numbers $E(\mathbf{r}_i + \delta\hat{\mathbf{e}}_\alpha) - E(\mathbf{r}_i - \delta\hat{\mathbf{e}}_\alpha)$ incurs an absolute roundoff error of $\sim |E|\epsilon_{\text{mach}}$, which gets amplified when divided by $2\delta$:
   $$\varepsilon_{\text{total}}(\delta) \approx c_2 \delta^2 + \frac{|E|\epsilon_{\text{mach}}}{2|F|\delta}$$
3. **Optimal Step Size $\delta^*$:** Minimizing $\varepsilon_{\text{total}}(\delta)$ with respect to $\delta$ yields $\delta^*_{\text{cen}} \propto \epsilon_{\text{mach}}^{1/3} \approx 6 \times 10^{-6}\text{ Å}$ (achieving $\varepsilon_{\min} \sim \epsilon_{\text{mach}}^{2/3} \approx 10^{-10}\text{–}10^{-11}$), whereas forward difference is optimal near $\delta^*_{\text{fwd}} \propto \epsilon_{\text{mach}}^{1/2} \approx 1.5 \times 10^{-8}\text{ Å}$ (achieving $\varepsilon_{\min} \sim 10^{-8}$). If your analytical force implementation has a bug, the error curve **plateaus at $\mathcal{O}(1)$** instead of dipping down to $10^{-10}$!


In [40]:
ws.render_section('concept_finite_difference_verification')

### Interactive Explorer: Finite-Difference Force Verification V-Curve

Sweep the log step size $\log_{10}(\delta)$ from $-12$ to $-1$ and compare the relative error of forward vs. central finite differences against the exact analytical force in IEEE-754 double precision.


**Key Takeaway:** Central finite differencing with $\delta \approx 10^{-5}\text{–}10^{-6}\text{ Å}$ agrees with a correct analytical gradient to $9\text{–}11$ significant digits ($\text{relative error} \sim 10^{-10}$). Making $\delta$ too small (e.g. $\delta = 10^{-11}\text{ Å}$) destroys accuracy due to $\epsilon_{\text{mach}}/\delta$ catastrophic cancellation.


In [41]:
ws.render_section('q31_central_difference_order')

**Q31 · easy** · 💡

Why does the central finite-difference formula $F_{i,\alpha}^{\text{cen}}(\delta) = -\frac{E(\mathbf{r}_i + \delta\hat{\mathbf{e}}_\alpha) - E(\mathbf{r}_i - \delta\hat{\mathbf{e}}_\alpha)}{2\delta}$ converge much faster with decreasing $\delta$ (in the truncation-dominated regime) than the one-sided forward difference?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Expand E(x + delta) and E(x - delta) in Taylor series up to delta^3 and subtract them.

</div>
</details>

- [ ] **(A)** The even-order Taylor terms ($\delta^2 \frac{d^2E}{dx^2}$) cancel out upon subtraction, leaving a leading truncation error of $\mathcal{O}(\delta^2)$ instead of $\mathcal{O}(\delta)$.
- [ ] **(B)** Central differencing eliminates IEEE-754 floating-point roundoff error for arbitrarily small $\delta$.
- [ ] **(C)** Central differencing requires only half as many energy evaluations as forward differencing.
- [ ] **(D)** The third derivative $d^3E/dx^3$ is identically zero for all interatomic potentials.

In [42]:
ws.render_section('q32_central_difference_harmonic_exactness')

**Q32 · medium** · 💡

Suppose a 1D bond potential is a pure quadratic parabola $E(x) = K_r(x - x_0)^2$ with $K_r = 15.0\text{ eV/Å}^2$ and $x_0 = 1.5\text{ Å}$. In exact arithmetic (ignoring floating-point roundoff), what is the truncation error $|F^{\text{cen}}(\delta) - F^{\text{analytic}}|$ of the central-difference formula at $x = 2.0\text{ Å}$ for a large step $\delta = 0.1\text{ Å}$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall that the leading truncation error of central difference is proportional to the third derivative d^3E/dx^3. What is d^3E/dx^3 for a quadratic polynomial?

</div>
</details>

_Numeric answer required._

In [43]:
ws.render_section('q33_optimal_step_scaling')

**Q33 · medium** · 💡

On a log-log plot of finite-difference force error versus step size $\delta$, why does the error **increase** with slope $-1$ ($\propto \delta^{-1}$) when $\delta$ is reduced below $\sim 10^{-6}\text{ Å}$ in double precision?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider what happens when E(x + delta) and E(x - delta) agree in almost all of their 16 significant decimal digits.

</div>
</details>

- [ ] **(A)** Subtractive cancellation of nearly equal floating-point numbers $E(x + \delta)$ and $E(x - \delta)$ produces a roundoff error of $\sim |E|\varepsilon_{\text{mach}}$, which grows as $\varepsilon_{\text{mach}} / \delta$ when divided by $2\delta$.
- [ ] **(B)** Taylor's theorem ceases to hold mathematically for $\delta < 10^{-6}\text{ Å}$.
- [ ] **(C)** The analytical force diverges as $\delta$ approaches zero.
- [ ] **(D)** The Born-Oppenheimer approximation breaks down at small numerical step sizes.

In [44]:
ws.render_section('q34_diagnosing_buggy_gradient')

**Q34 · medium** · 💡

You implement the Blondel–Karplus dihedral force routine and run a central finite-difference gradient check across $\delta \in [10^{-8}, 10^{-2}]\text{ Å}$. For atoms $1$ and $4$, the relative error drops to $10^{-10}$ at $\delta = 10^{-5}\text{ Å}$, but for atoms $2$ and $3$, the relative error stays flat at $0.38$ across all $\delta \in [10^{-7}, 10^{-3}]\text{ Å}$. What is the most likely diagnosis?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compare which terms appear in nabla_{r_2} phi and nabla_{r_3} phi on page 3 of the handout that do NOT appear in nabla_{r_1} phi and nabla_{r_4} phi.

</div>
</details>

- [ ] **(A)** There is an algebraic or sign error in the internal-atom projection terms $\frac{\mathbf{r}_{12}\cdot\mathbf{r}_{32}}{n_{123}^2 r_{23}}\mathbf{n}_{123}$ or $\frac{\mathbf{r}_{43}\cdot\mathbf{r}_{23}}{n_{432}^2 r_{23}}\mathbf{n}_{432}$ that only enter $\nabla_{\mathbf{r}_2}\phi$ and $\nabla_{\mathbf{r}_3}\phi$.
- [ ] **(B)** The step size $\delta = 10^{-5}\text{ Å}$ is too small for internal atoms.
- [ ] **(C)** The scalar derivative $dE/d\phi$ has the wrong sign.
- [ ] **(D)** Central finite differencing cannot be applied to internal atoms of a dihedral.

In [45]:
ws.render_section('q35_central_difference_calculation_numeric')

**Q35 · easy** · 💡

During a gradient test on atom $3$'s $z$-coordinate with perturbation $\delta = 0.001\text{ Å}$, your potential energy function returns $E(z_3 + \delta) = -14.2036\text{ eV}$ and $E(z_3 - \delta) = -14.1984\text{ eV}$. What is the central finite-difference estimate of the force component $F_{3,z}$ (in $\text{eV/Å}$)? *(Include the correct algebraic sign.)*


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate F_{3,z} = -(E(z_3 + delta) - E(z_3 - delta)) / (2 * delta), paying careful attention to the minus sign in F = -dE/dz.

</div>
</details>

_Numeric answer required._

In [46]:
ws.render_section('q36_synthesis_all_terms_properties')

**Q36 · hard** · 💡

Synthesizing all three covalent bonded derivative derivations from `Interatomic-Derivatives.pdf`, which of the following geometric statements about the analytical forces are **exact identities** for every non-degenerate atomic configuration? *(Select all that apply.)*


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check the direction of 2-body bond forces, 3-body terminal angle forces, 4-body terminal dihedral forces, and the sum of forces for each interaction.

</div>
</details>

- [ ] **(A)** For a 2-body bond stretch $E_{\text{bond}}(r_{12})$, $\mathbf{F}_1$ and $\mathbf{F}_2$ are collinear with the bond vector $\mathbf{r}_{12}$ and satisfy $\mathbf{F}_1 + \mathbf{F}_2 = 0$.
- [ ] **(B)** For a 3-body bond angle $E_{\text{angle}}(\theta_{123})$ with central vertex atom 2, $\mathbf{F}_1$ lies in the (1-2-3) plane strictly perpendicular to $\mathbf{r}_{12}$, $\mathbf{F}_3$ lies in the (1-2-3) plane strictly perpendicular to $\mathbf{r}_{32}$, and $\mathbf{F}_2 = -(\mathbf{F}_1 + \mathbf{F}_3)$.
- [ ] **(C)** For a 4-body torsion $E(\phi_{1234})$, $\mathbf{F}_1$ is parallel to the plane normal $\mathbf{n}_{123}$, $\mathbf{F}_4$ is parallel to the plane normal $\mathbf{n}_{432}$, all four forces are perpendicular to the central bond $\mathbf{r}_{23}$, and $\mathbf{F}_1 + \mathbf{F}_2 + \mathbf{F}_3 + \mathbf{F}_4 = 0$.
- [ ] **(D)** For a 3-body bond angle $E_{\text{angle}}(\theta_{123})$, the force $\mathbf{F}_2$ on the central vertex atom is always zero.

In [47]:
ws.render_section('q37_opls_nonbonded_intramolecular_scaling')

**Q37 · medium** · 💡

In the OPLS-AA energy expression on page 1 of `Interatomic-Derivatives.pdf`, why are non-bonded Coulomb and Lennard-Jones interactions excluded between directly bonded ($1\text{–}2$) and angle-sharing ($1\text{–}3$) atom pairs (and scaled by $0.5$ for $1\text{–}4$ dihedral pairs)?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider what would happen at r_12 ~ 1.5 A if a stiff (sigma/r)^12 Lennard-Jones core repulsion were added on top of the covalent harmonic bond E_bond = K_r(r - r_eq)^2.

</div>
</details>

- [ ] **(A)** Because covalent interactions at short 1-2 and 1-3 distances are already explicitly modeled by $E_{\text{bond}}$ and $E_{\text{angle}}$; evaluating unscaled $(\sigma/r)^{12}$ Lennard-Jones repulsion at covalent bond lengths ($\sim 1.5\text{ Å} \ll \sigma \sim 3.5\text{ Å}$) would produce enormous spurious repulsive forces.
- [ ] **(B)** Because the gradient $\nabla_{\mathbf{r}_1}|\mathbf{r}_1 - \mathbf{r}_2|$ is undefined for covalently bonded atoms.
- [ ] **(C)** Because covalently bonded atoms carry zero mass in MD.
- [ ] **(D)** Because Lennard-Jones forces violate Newton's third law for bonded pairs.

In [48]:
ws.render_section('q38_total_force_accumulation_loop')

**Q38 · medium** · 💡

In an all-atom MD force evaluation routine, a single carbon atom $i$ participates simultaneously in 4 covalent bonds, 6 bond angles, 9 dihedral torsions, and hundreds of non-bonded pairs. How is the net force $\mathbf{F}_i$ on atom $i$ obtained from these individual interaction derivatives?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall the linearity of the gradient operator nabla_{r_i} applied to a sum of energy terms E_total = E_bond + E_angle + E_torsion + E_nb.

</div>
</details>

- [ ] **(A)** By linearity of the gradient operator, $\mathbf{F}_i$ is the vector sum of the negative gradients accumulated from every bond, angle, torsion, and non-bonded pair in which atom $i$ participates.
- [ ] **(B)** Only the stiffest term (the bond-stretching gradient) is kept for each atom, while angle and torsion forces are discarded.
- [ ] **(C)** The forces from different terms are multiplied together rather than added.
- [ ] **(D)** Each atom can only receive a force from at most one internal coordinate per timestep.

In [ ]:
ws.show_record()

In [ ]:
ws.finalize()